# Hiding the Minimum Threat in a Small Network

**Team 8 — Part A.** A tiny fully-connected MNIST network, and a quick check of how much could be
hidden in its weights — enough for the smallest realistic "threat", the 68-byte EICAR test file.
No embedding yet: this only measures capacity from the model's shape.

Reference: Wang et al., *EvilModel*, 2021 ([arXiv:2107.08590](https://arxiv.org/abs/2107.08590)).

In [8]:
import torch
import torch.nn as nn


class Net(nn.Module):
    # Small fully-connected MNIST classifier: 784 -> 32 -> 10.
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28 * 28, 32)
        self.fc2 = nn.Linear(32, 10)

    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(self.flatten(x))))


model = Net()
print(model)

Net(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=784, out_features=32, bias=True)
  (fc2): Linear(in_features=32, out_features=10, bias=True)
)


In [9]:
# "Minimum threat": EICAR, the standard harmless antivirus TEST file (68 bytes).
# Every AV flags it as a threat, so it is the smallest realistic thing to hide.
# We only need its SIZE here -- no binary, no embedding.
MIN_THREAT_BYTES = 68          # EICAR test file


def weight_count(model):
    # Total number of weight values (biases excluded).
    return sum(p.numel() for name, p in model.named_parameters() if name.endswith("weight"))


def capacity_bytes(model, n_bits=1):
    # Bytes we could hide using the lowest n_bits of every weight.
    return weight_count(model) * n_bits // 8


w = weight_count(model)
cap = capacity_bytes(model, n_bits=1)
need = MIN_THREAT_BYTES * 8     # weights needed at 1 bit each

print(f"weights in model          : {w:,}")
print(f"capacity at 1 bit/weight  : {cap:,} bytes")
print(f"minimum threat (EICAR)    : {MIN_THREAT_BYTES} bytes")
print(f"weights needed to hide it : {need:,} (at 1 bit/weight)")
print()
print("result:", "fits easily" if MIN_THREAT_BYTES <= cap else "does NOT fit")

weights in model          : 25,408
capacity at 1 bit/weight  : 3,176 bytes
minimum threat (EICAR)    : 68 bytes
weights needed to hide it : 544 (at 1 bit/weight)

result: fits easily


Even this small network has far more low-bit room than the minimum threat needs: hiding 68 bytes
takes only 544 weights (1 bit each). The next step is to actually embed it and check whether the
model still classifies correctly.